# data ingestion from JDBC - microsoft SQL server driver

In [0]:

from common_utils.logging import logger

logger = get_logger('ds2b_sqlserver')

url = "jdbc:sqlserver://rivadata.database.windows.net:1433;databaseName=batch2;encrypt=true;trustServerCertificate=true;loginTimeout=90" 

user = "rivadata" 

password = dbutils.secrets.get(scope='retail-platform-dev', key='sqlserver_password')

dbtable = "retail.customers"

df = spark.read.format('jdbc')\
    .option('url',url)\
    .option('dbtable',dbtable)\
    .option('user', user)\
    .option('password', password)\
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')\
    .load()

display(df)


# write to new catalog
# we have to fetch the date--automatically-real time -not hardcoded

#import pyspark.sql.functions as F


from datetime import date
run_date = date.today().isoformat()

#run_date = '2026-09-07'

target_path = f"/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date={run_date}"

df.write.mode('overwrite')\
    .option('header','true')\
    .option('quoteAll','true')\
        .option('escape','"')\
            .csv(target_path)



In [0]:
# write to new catalog
# we have to fetch the date--automatically-real time -not hardcoded

#import pyspark.sql.functions as F


from datetime import date
run_date = date.today().isoformat()

#run_date = '2026-09-07'

target_path = f"/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date={run_date}"

df.write.mode('overwrite')\
    .option('header','true')\
    .option('quoteAll','true')\
        .option('escape','"')\
            .csv(target_path)

In [0]:

from common_utils.logging import get_logger

logger = get_logger('ds2b_sqlserver')

In [0]:
# read data from raw_data volume - done
# add audit columnms -> 2 columns - last_update_ts, file_path
# define target
# write data in delta table


#logging
'''
basic - print statement

advanced - python logger

'''

from common_utils.logging import get_logger

logger = get_logger('ds2b_sqlserver')

import pyspark.sql.functions as F 


logger.info("Defining raw path")
raw_path = "/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date=2026-09-07/"
logger.info(f"raw path given is {raw_path}")




logger.info("reading data")
df = spark.read\
        .format("csv")\
        .option("header","true")\
        .option("inferSchema", "true")\
        .load(raw_path)

logger.info("sample data",df.show())

logger.info("adding audit columns")
df = df.withColumn("last_update_ts", F.current_timestamp() )\
        .withColumn("file_path", F.col("_metadata.file_path"))

target_path = "retaildataplatform.bronze.sqlserver_customers"
logger.info(f"target path is {target_path}")

logger.warning("writing data")
df.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(target_path)

logger.info("Data writeen Successfully at", target_path)
